In [1]:
import sys
import os
# 添加 server 目录到 Python 路径
sys.path.append(os.curdir + "/server")
sys.path

['/opt/homebrew/Cellar/python@3.12/3.12.10/Frameworks/Python.framework/Versions/3.12/lib/python312.zip',
 '/opt/homebrew/Cellar/python@3.12/3.12.10/Frameworks/Python.framework/Versions/3.12/lib/python3.12',
 '/opt/homebrew/Cellar/python@3.12/3.12.10/Frameworks/Python.framework/Versions/3.12/lib/python3.12/lib-dynload',
 '',
 '/Users/weibo/code/myself/live2d-assistant/venv/lib/python3.12/site-packages',
 './server']

In [2]:
from agent import Agent, Assistant
from client import MCPClient, MCPClientConfig, MCPServerConfig, LLMConfig
from langchain_openai import ChatOpenAI
from langchain_community.tools.ddg_search import DuckDuckGoSearchRun
from langchain_community.tools.searx_search.tool import SearxSearchRun
from langchain_community.utilities import SearxSearchWrapper
from langgraph.graph import StateGraph

In [3]:
llm = ChatOpenAI(model='deepseek-chat', base_url="https://api.deepseek.com/", api_key='sk-a85fd3f25372446ea811be805115c061')
ddg_tool = DuckDuckGoSearchRun()
search_tool= SearxSearchRun(wrapper=SearxSearchWrapper(searx_host="http://127.0.0.1:8001", k=3))

In [4]:
config = MCPClientConfig(
    mcp_servers=[
        MCPServerConfig(
            name="music_player",
            transport="sse",
            url="http://localhost:8000/sse"
        )
    ],
    llm=LLMConfig(  
        provider="ollama",
        api_key="ollama",
        base_url="http://localhost:11434/v1",
        sys_prompt="你是一个经验丰富的助手",
    ),
)

mcp_client = MCPClient(config)
await mcp_client.connect_to_servers()
tools = await mcp_client.list_tools("music_player")
tools.tools

[Tool(name='play_music', description='\n播放音乐\n\nparams: \n    song_name: 歌曲名或关键字\n\n', inputSchema={'properties': {'song_name': {'title': 'Song Name', 'type': 'string'}}, 'required': ['song_name'], 'title': 'play_musicArguments', 'type': 'object'}, annotations=None),
 Tool(name='isPlaying', description='\ncheck if playing\n\nreturn: \n    Yes if playing, No if not playing\n', inputSchema={'properties': {}, 'title': 'isPlayingArguments', 'type': 'object'}, annotations=None),
 Tool(name='stopplay', description='\n停止播放音乐\n\n返回: \n    播放状态: 已停止\n', inputSchema={'properties': {}, 'title': 'stopplayArguments', 'type': 'object'}, annotations=None),
 Tool(name='pauseplay', description='\n暂停音乐播放\n\n返回: \n    播放状态: 已暂停\n', inputSchema={'properties': {}, 'title': 'pauseplayArguments', 'type': 'object'}, annotations=None),
 Tool(name='unpauseplay', description='\n恢复音乐播放\n\n返回：\n    播放状态: 已恢复播放\n', inputSchema={'properties': {}, 'title': 'unpauseplayArguments', 'type': 'object'}, annotations=None)]

In [5]:
import asyncio
import threading
from typing import Any

from langchain_core.tools import BaseTool, StructuredTool
from langchain_mcp_adapters.tools import NonTextContent, _convert_call_tool_result
from mcp import ClientSession

from mcp.types import (
    Tool as MCPTool
)

_loop = asyncio.new_event_loop()

_thr = threading.Thread(target=_loop.run_forever, name="Async Runner",
                        daemon=True)

# This will block the calling thread until the coroutine is finished.
# Any exception that occurs in the coroutine is raised in the caller
def run_async(coro):  # coro is a couroutine, see example
    """
    Ref: https://stackoverflow.com/a/74710015
    """
    if not _thr.is_alive():
        _thr.start()
    future = asyncio.run_coroutine_threadsafe(coro, _loop)
    return future.result()

def new_convert_mcp_tool_to_langchain_tool(
    session: ClientSession,
    tool: MCPTool,
) -> BaseTool:
    """Convert an MCP tool to a LangChain tool.

    NOTE: this tool can be executed only in a context of an active MCP client session.

    Args:
        session: MCP client session
        tool: MCP tool to convert

    Returns:
        a LangChain tool
    """

    async def acall_tool(
        **arguments: dict[str, Any],
    ) -> tuple[str | list[str], list[NonTextContent] | None]:
        print(f"调用工具 {tool.name} 参数 {arguments}")
        call_tool_result = await session.call_tool(tool.name, arguments)
        print(f"工具 {tool.name} 返回结果 {call_tool_result}")
        return _convert_call_tool_result(call_tool_result)

    def call_tool(
        **arguments: dict[str, Any],
    ) -> tuple[str | list[str], list[NonTextContent] | None]:
        return run_async(acall_tool(**arguments))

    return StructuredTool(
        name=tool.name,
        description=tool.description or "",
        args_schema=tool.inputSchema,
        func=call_tool,  #
        # coroutine=acall_tool,
        response_format="content_and_artifact",
    )


In [6]:
assistant_map = {
    "research": Assistant(
        name="research",
        description="research 助手可以进行网络搜索。",
        tools=[search_tool],
        prompt="""你是一个网络搜索助手，请根据用户的问题进行网络搜索，并返回搜索结果。
        """
    ),
    "decision_maker": Assistant(
        name="decision_maker",
        description="进行规划和决策任务，并将任务分配给合适的助手。",
        tools=[],
        prompt="""你是一个决策助手，负责根据用户请求决策应该如何执行。
你应该遵守如下的处理逻辑
1. 如果用户请求的是一个一般性问题，请直接回答
2. 如果用户需要你完成某个任务，你需要将任务分配给其他助手
"""
    ),
    "music_player": Assistant(
        name="music_player",
        description="music_player 助手可以播放指定的音乐。",
        tools=[new_convert_mcp_tool_to_langchain_tool(await mcp_client.get_client_session("music_player"), tool) for tool in tools.tools],
        prompt="""你是一个音乐播放助手，请根据用户的问题进行音乐播放，并返回播放结果。
        """
    ),
}
agent = Agent(
    llm=llm,
    assistant_map=assistant_map,
    start_node="decision_maker"
)

graph: StateGraph = agent.create_workflow()

创建节点 research
创建节点 decision_maker
创建节点 music_player


In [7]:
from IPython.display import Image, display
from langchain_core.runnables.graph import MermaidDrawMethod
 
g = graph.get_graph()
with open("graph.md", "w") as f:
    f.write(g.draw_mermaid())

In [8]:
events = graph.astream(
    {
        "messages": [
            (
                "user",
                "搜索伍佰最热门的一首歌并播放，播放完成后，退出",
            )
        ],
    },
    # Maximum number of steps to take in the graph
    {"recursion_limit": 150},
)
async for s in events:
    print(s)
    print("----")

节点 decision_maker 返回结果 {'messages': [HumanMessage(content='搜索伍佰最热门的一首歌并播放，播放完成后，退出', additional_kwargs={}, response_metadata={}, id='c58dafb5-0030-4b31-b4f5-53ce677181c1'), AIMessage(content='思考：用户请求需要完成两个任务，首先是搜索伍佰最热门的一首歌，然后播放该歌曲。这需要先进行网络搜索，然后使用音乐播放功能。\n\n执行：首先需要调用research助手搜索伍佰最热门的一首歌，然后让music_player助手播放搜索结果。\n\nGOTO: research', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 65, 'prompt_tokens': 378, 'total_tokens': 443, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 320}, 'prompt_cache_hit_tokens': 320, 'prompt_cache_miss_tokens': 58}, 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_8802369eaa_prod0425fp8', 'id': '32638629-a31b-4fbf-9ff5-5b113e06d5b4', 'service_tier': None, 'finish_reason': 'stop', 'logprobs': None}, id='run--4d6ae987-9d57-456a-a133-4af358d8be81-0', usage_metadata={'input_tokens': 378, 'output_tokens': 65, 'total_tokens': 443, 'input_token_details': {'cache_re

CancelledError: 